In [ ]:
# According to Kaggle, my model submission result is 0.127 and 13463.00913 (RMSE) 

In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


pd.pandas.set_option('display.max_columns', None)
pd.pandas.set_option('display.max_rows', None)

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter("ignore", category=ConvergenceWarning)


def data():
    train_ = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
    test_ = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
    dataframe = pd.concat([train_, test_], ignore_index=True)
    return dataframe, train_, test_
df, train, test = data()


def info_(dataframe):
    print(dataframe.shape)

for i in [df, train, test]:
    info_(i)


# NİHAİ DF ( CATBOOST EN ÖNEMLİ DEĞİŞKENLER, ORADA 70 TANE VARDI AMA ONE HOT YAPTIĞIMIZ İÇİN TÜREYEN DEĞİŞKENLERDİ)
list = ['OverallQual', 'GrLivArea', 'GarageCars', '1stFlrSF', 'TotalBsmtSF', 'BsmtFinSF1', 'LotArea', '2ndFlrSF',
        'YearRemodAdd', 'FullBath', 'GarageArea', 'YearBuilt', 'OverallCond', 'Fireplaces', 'LotFrontage',
        'BsmtFullBath', 'HalfBath', 'OpenPorchSF', 'WoodDeckSF', 'GarageYrBlt', 'Id', 'BedroomAbvGr', 'BsmtUnfSF',
        'ScreenPorch', 'MoSold', 'MasVnrArea', 'KitchenAbvGr', 'YrSold', 'TotRmsAbvGrd', 'MSSubClass', 'EnclosedPorch',
        "ExterCond", "KitchenQual", "Neighborhood", "CentralAir", "MSZoning", "SaleCondition", "Condition1",
        "ExterQual", "LandContour", "Functional", "Exterior2nd", "BsmtFinType1", "Exterior1st", "BsmtQual",
        "GarageType", "LotConfig", "HouseStyle", "SaleType", "HeatingQC", "BsmtExposure", "GarageFinish", "MasVnrType",
        "RoofStyle", "GarageCond", "RoofMatl", 'SalePrice']

df = df[list]
print(df.shape)


# BU ÇALIŞMADA DEĞİŞKENLERİ 4 LİSTE HALİNDE İNCELEDİM: NOM-ORD-NUMLIST-NUMLIST2.
# BU LİSTELERE GÖRE FARKLI ÖN İŞLEMELER YAPTIM

obj_list = [col for col in df.columns if df[col].dtypes == "O"]

ord_list = []
for i in obj_list:
    if df[i].str.contains('Gd', regex=True).any() == True:
        ord_list.append(i)

nom_list = []
for i in obj_list:
    if i not in ord_list:
        nom_list.append(i)

num_list = [] # bildiğimiz numerikler
num_list2 = [] # numerik görünümlü kategorikler
for i in df.columns:
    if df[i].dtypes != "O":
        if df[i].nunique() < 15:
            num_list2.append(i)
        elif df[i].nunique() >= 15:
            num_list.append(i)

# TÜM DEĞİŞKENLERİ ALDIM MI, KONTROLÜ
len(num_list2) + len(num_list) + len(nom_list) + len(ord_list)

# ORDINAL
# ORDİNAL DEĞİŞKENLERDE KAGGLE DA OLAN EX-GD-TA.. SIRALAMASINA GÖRE LABEL ENCODER YAPTIM.
for i in df[ord_list].columns:
    for k in range(len(df)):
        if i not in "BsmtExposure":
            if df.loc[k, i] == "Ex":
                df.loc[k, i] = 6
            elif df.loc[k, i] == "Gd":
                df.loc[k, i] = 5
            elif df.loc[k, i] == "TA":
                df.loc[k, i] = 4
            elif df.loc[k, i] == "Fa":
                df.loc[k, i] = 3
            elif df.loc[k, i] == "Po":
                df.loc[k, i] = 2
            else:
                df.loc[k, i] = 1

# BU DEĞİŞKENDEKİ KAGGLE SIRALAMASI FARKLI İDİ: GD-AV-MN GİBİ. BUNU AYRICA DÜZENLEDİM
df.loc[df["BsmtExposure"] == "Gd", "BsmtExposure"] = 5
df.loc[df["BsmtExposure"] == "Av", "BsmtExposure"] = 4
df.loc[df["BsmtExposure"] == "Mn", "BsmtExposure"] = 3
df.loc[df["BsmtExposure"] == "No", "BsmtExposure"] = 2
df["BsmtExposure"] = df["BsmtExposure"].fillna(1)

# NOMINAL

# NA LERE XNA ATAMA
# ( BUNU YAPMAMIN NEDENİ: BAZI İŞLEMLER VERİ SETİNDE NA OLUNCA GERÇEKLEŞMİYOR, BU NEDENLE BİR İSİM KOYDUM)

for i in df[nom_list]:
    df[i].replace(to_replace=np.nan, value='XNA', regex=True, inplace=True)

# RARE ATAMA
# ALT SINIFI 0.05'TEN KÜÇÜK OLANLARA RARE ATADIM.
for i in df[nom_list]:
    for k in df[i].unique():
        if df.loc[df[i] == k, i].shape[0] / len(df) < 0.05:
            df.loc[df[i] == k, i] = "Rare"


# ONE HOT SONUCUNDA SHAPE: (2919,94). 57 SATIRDAN 94 SÜTUN ÇIKTI.
def one_hot_encoder(dataframe, categorical_cols):
    dataframe = pd.get_dummies(dataframe, columns=categorical_cols, drop_first=True)
    return dataframe


df2 = one_hot_encoder(df, nom_list)

# NUM LİST2

# MEDIAN ATAMA
# BİR DEĞİŞKENDE 10'DAN AZ EKSİK DEĞER VARSA MEDİAN ATADIM.
for i in df2[num_list2]:
    if df2[i].isnull().sum() < 10:
        df2[i] = df2[i].fillna(df2[i].median())


# ALT SINIF ANALİZİ YAPTIM. SONRASINDA TEK TEK DEĞİŞKENLERİ İNCELEDİM.

def class_analyze(dataframe, target, liste):
    for var in liste:
        print(var, ":", len(dataframe[var].value_counts()))
        print(pd.DataFrame({"COUNT": dataframe[var].value_counts(),
                            "RATIO": dataframe[var].value_counts() / len(dataframe),
                            "TARGET_MEDIAN": dataframe.groupby(var)[target].median()})
              .sort_values(by="TARGET_MEDIAN", ascending=False), end="\n\n\n")


class_analyze(df2, "SalePrice", num_list2)

# İNCELEDİĞİM DEĞİŞKENLER AŞAĞIDAKİ GİBİ. SALEPRİCE DEĞERLERİNE GÖRE SIRALAMA YAPARAK LABEL YAPTIM.
# OverallQual ok
# GarageCars
df2.loc[df["GarageCars"] == 3, "GarageCars"] = 5
df2.loc[df["GarageCars"] == 2, "GarageCars"] = 3
df2.loc[df["GarageCars"] == 1, "GarageCars"] = 2
df2.loc[df["GarageCars"] == 0, "GarageCars"] = 1
# FullBath ok
# OverallCond
df2.loc[df["OverallCond"] == 5, "OverallCond"] = 9
df2.loc[df["OverallCond"] == 9, "OverallCond"] = 8
df2.loc[df["OverallCond"] == 8, "OverallCond"] = 5
# Fireplaces ok
# BsmtFullBath 
df2.loc[df["BsmtFullBath"] == 2, "BsmtFullBath"] = 3
df2.loc[df["BsmtFullBath"] == 3, "BsmtFullBath"] = 2
# HalfBath  
df2.loc[df["HalfBath"] == 1, "HalfBath"] = 2
df2.loc[df["HalfBath"] == 2, "HalfBath"] = 1
# BedroomAbvGr
df2.loc[df["BedroomAbvGr"] == 0, "BedroomAbvGr"] = 8
df2.loc[df["BedroomAbvGr"] == 8, "BedroomAbvGr"] = 7
df2.loc[df["BedroomAbvGr"] == 4, "BedroomAbvGr"] = 6
df2.loc[df["BedroomAbvGr"] == 3, "BedroomAbvGr"] = 5
df2.loc[df["BedroomAbvGr"] == 5, "BedroomAbvGr"] = 4
df2.loc[df["BedroomAbvGr"] == 1, "BedroomAbvGr"] = 3
df2.loc[df["BedroomAbvGr"] == 6, "BedroomAbvGr"] = 2
df2.loc[df["BedroomAbvGr"] == 2, "BedroomAbvGr"] = 1
# MoSold
df2.loc[df["MoSold"] == 9, "MoSold"] = 12
df2.loc[df["MoSold"] == 12, "MoSold"] = 11
df2.loc[df["MoSold"] == 8, "MoSold"] = 10
df2.loc[df["MoSold"] == 2, "MoSold"] = 9
df2.loc[df["MoSold"] == 11, "MoSold"] = 8
df2.loc[df["MoSold"] == 3, "MoSold"] = 7
df2.loc[df["MoSold"] == 7, "MoSold"] = 6
df2.loc[df["MoSold"] == 6, "MoSold"] = 5
df2.loc[df["MoSold"] == 10, "MoSold"] = 4
df2.loc[df["MoSold"] == 5, "MoSold"] = 3
df2.loc[df["MoSold"] == 1, "MoSold"] = 2
df2.loc[df["MoSold"] == 4, "MoSold"] = 1
# KitchenAbvGr
df2.loc[df["KitchenAbvGr"] == 1, "KitchenAbvGr"] = 3
df2.loc[df["KitchenAbvGr"] == 2, "KitchenAbvGr"] = 2
df2.loc[df["KitchenAbvGr"] == 0, "KitchenAbvGr"] = 1
df2.loc[df["KitchenAbvGr"] == 3, "KitchenAbvGr"] = 0
# TotRmsAbvGrd
df2.loc[df["TotRmsAbvGrd"] == 11, "TotRmsAbvGrd"] = 14
df2.loc[df["TotRmsAbvGrd"] == 10, "TotRmsAbvGrd"] = 13
df2.loc[df["TotRmsAbvGrd"] == 9, "TotRmsAbvGrd"] = 12
df2.loc[df["TotRmsAbvGrd"] == 8, "TotRmsAbvGrd"] = 11
df2.loc[df["TotRmsAbvGrd"] == 12, "TotRmsAbvGrd"] = 10
df2.loc[df["TotRmsAbvGrd"] == 14, "TotRmsAbvGrd"] = 9
df2.loc[df["TotRmsAbvGrd"] == 7, "TotRmsAbvGrd"] = 8
df2.loc[df["TotRmsAbvGrd"] == 6, "TotRmsAbvGrd"] = 7
df2.loc[df["TotRmsAbvGrd"] == 5, "TotRmsAbvGrd"] = 6
df2.loc[df["TotRmsAbvGrd"] == 4, "TotRmsAbvGrd"] = 5
df2.loc[df["TotRmsAbvGrd"] == 3, "TotRmsAbvGrd"] = 4
df2.loc[df["TotRmsAbvGrd"] == 2, "TotRmsAbvGrd"] = 3
df2.loc[df["TotRmsAbvGrd"] == 13, "TotRmsAbvGrd"] = 2
df2.loc[df["TotRmsAbvGrd"] == 15, "TotRmsAbvGrd"] = 1


# NUM LİST
df2.loc[df2["GrLivArea"] > 4000, "GrLivArea"] = 3800
df2.loc[df2["1stFlrSF"] > 3500, "1stFlrSF"] = 3500
df2.loc[df2["TotalBsmtSF"] > 4000, "TotalBsmtSF"] = 3200
df2.loc[df2["BsmtFinSF1"] > 3000, "BsmtFinSF1"] = 1270  # 0.95 quantile baskıladım.
df2.loc[df2["2ndFlrSF"] > 1750, "2ndFlrSF"] = 1750
df2.loc[df2["GarageArea"] > 1330, "GarageArea"] = 1330
df2.loc[df2["LotFrontage"] > 200, "LotFrontage"] = 200
df2.loc[df2["WoodDeckSF"] > 736, "WoodDeckSF"] = 736
df2.loc[df["GarageArea"] == 0, "GarageYrBlt"] = 0  # Garage Area'ları yok ama yapıldığı yıl 1979, bunun yerine 0 verdim.
df2.loc[df2["BsmtUnfSF"] >= 2122, "BsmtUnfSF"] = 2122
df2.loc[df2["MasVnrArea"] > 1230, "MasVnrArea"] = 1230


# VERİ SETİNDEKİ DİĞER NA'LAR

# EKSİK DEĞER SAYISI 30'DAN AZ OLANLARA MEDIAN ATA
for i in df2.columns:
    if df2[i].isnull().sum() < 30:
        df2[i] = df2[i].fillna(df2[i].median())


# NA OLAN SALEPRICE DIŞINDA SADECE LOTFRONTAGE KALDI.

# NA LERE XNA ATAMA
df2["LotFrontage"].replace(to_replace=np.nan, value=-99, regex=True, inplace=True)

# LotFrontage' da NA olanların SalePrice ortalaması 172k
df2.loc[df2["LotFrontage"] == -99, "SalePrice"].median()

# Genelde SalePrice ortalaması 163k.
df2["SalePrice"].median()

# İKİ DEĞER BİRBİRİNE ÇOK YAKIN. BU NEDENLE MEDIAN ATAYABILIRIM.
df2.loc[df2["LotFrontage"] == -99, "LotFrontage"] = 65

#df2.loc[(df2["SalePrice"]<=175000)&(df2["SalePrice"]>=165000), "LotFrontage"].median()



# DİĞER DEĞİŞKENLER

df3 = df2.copy()
df3.loc[df3["OverallQual"] < 2, "OverallQual"] = 2
df3.loc[df3["GarageCars"] > 4, "GarageCars"] = 4
df3.loc[(df3["OverallQual"] == 2) | (df3["OverallQual"] == 3), "OverallQual"] = 3
# SalePrice 163k, genelde 163k salepricelıların garaj built değeri 2005
df3.loc[df3["GarageYrBlt"] == 2207, "GarageYrBlt"] = 2005
df3.loc[df3["Fireplaces"] == 4, "Fireplaces"] = 0
df3.loc[df3["Fireplaces"] == 3, "Fireplaces"] = 2
# NEW_1
df3["new_area"] = df3["GrLivArea"] + df3["GarageArea"]
# NEW_2  #1K'YA YAKIN DÜŞTÜ
df3["new_home"] = df3["YearBuilt"]
df3.loc[df3["new_home"] == df3["YearRemodAdd"], "new_home"] = 0
df3.loc[df3["new_home"] != df3["YearRemodAdd"], "new_home"] = 1
# NEW_3  #Banyo sayısı toplamları
df3["new_bath"] = df3["FullBath"] + (df3["HalfBath"] * 0.5)


# CATBOOST MODEL : 20.18K
X = df3.loc[:1459, :].drop(["SalePrice", "Id"], axis=1)
y = df3.loc[:1459, "SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=46)

catboost_model = CatBoostRegressor()
catboost_model = catboost_model.fit(X_train, y_train)
y_pred = catboost_model.predict(X_test)
print(np.sqrt(mean_squared_error(y_pred, y_test)))


# TO CSV , KAGGLE'A YÜKLEMEK İÇİN
df107 = pd.DataFrame({"Id": df3.loc[1460:, "Id"],
                      "SalePrice": catboost_model.predict(df3.loc[1460:, :].drop(["SalePrice", "Id"], axis=1))})

df107.to_csv('deneme7.csv', index=False)



# Robust Scaler
df4=df3.copy()
from sklearn.preprocessing import RobustScaler
robust_list=[col for col in df4.columns if col not in "Id" and col not in "SalePrice"]

col=robust_list
x_transformed=pd.DataFrame(RobustScaler().fit(df4.loc[:,robust_list]).transform(df4.loc[:,robust_list]), columns=col)
x_transformed.head()

non_robust_list=[col for col in df4.columns if col not in robust_list]
df5=pd.concat([df4.loc[:,non_robust_list], x_transformed], axis=1)
df5.head(2)


# CATBOOST MODEL : 20.07K
X = df5.loc[:1459, :].drop(["SalePrice", "Id"], axis=1)
y = df5.loc[:1459, "SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=46)

catboost_model = CatBoostRegressor()
catboost_model = catboost_model.fit(X_train, y_train)
y_pred = catboost_model.predict(X_test)
print(np.sqrt(mean_squared_error(y_pred, y_test)))


# TO CSV , KAGGLE'A YÜKLEMEK İÇİN

df108 = pd.DataFrame({"Id": df5.loc[1460:, "Id"],
                      "SalePrice": catboost_model.predict(df5.loc[1460:, :].drop(["SalePrice", "Id"], axis=1))})

df108.to_csv('deneme8.csv', index=False)




(2919, 81)
(1460, 81)
(1459, 80)
(2919, 57)
OverallQual : 10
    COUNT     RATIO  TARGET_MEDIAN
10     31  0.010620       432390.0
9     107  0.036656       345000.0
8     342  0.117163       269750.0
7     600  0.205550       200141.0
6     731  0.250428       160000.0
5     825  0.282631       133000.0
4     226  0.077424       108000.0
3      40  0.013703        86250.0
2      13  0.004454        60000.0
1       4  0.001370        50150.0


GarageCars : 6
     COUNT     RATIO  TARGET_MEDIAN
3.0    374  0.128126       295000.0
4.0     16  0.005481       200000.0
2.0   1595  0.546420       177750.0
1.0    776  0.265844       128000.0
0.0    157  0.053786       100000.0
5.0      1  0.000343            NaN


FullBath : 5
   COUNT     RATIO  TARGET_MEDIAN
3     64  0.021925       320000.0
2   1530  0.524152       196750.0
0     12  0.004111       145000.0
1   1309  0.448441       132375.0
4      4  0.001370            NaN


OverallCond : 9
   COUNT     RATIO  TARGET_MEDIAN
5   1645  0.56

100:	learn: 23327.7342431	total: 387ms	remaining: 3.44s
101:	learn: 23222.2590680	total: 391ms	remaining: 3.44s
102:	learn: 23134.1117163	total: 394ms	remaining: 3.43s
103:	learn: 23018.5106918	total: 397ms	remaining: 3.42s
104:	learn: 22887.5939399	total: 400ms	remaining: 3.41s
105:	learn: 22764.6227022	total: 403ms	remaining: 3.4s
106:	learn: 22646.4929418	total: 407ms	remaining: 3.39s
107:	learn: 22558.5760398	total: 410ms	remaining: 3.38s
108:	learn: 22454.9472640	total: 413ms	remaining: 3.38s
109:	learn: 22362.4833627	total: 417ms	remaining: 3.37s
110:	learn: 22248.7090604	total: 420ms	remaining: 3.36s
111:	learn: 22154.6847898	total: 423ms	remaining: 3.35s
112:	learn: 22055.0405399	total: 426ms	remaining: 3.35s
113:	learn: 21981.9964717	total: 430ms	remaining: 3.34s
114:	learn: 21898.2176259	total: 433ms	remaining: 3.33s
115:	learn: 21798.8277661	total: 436ms	remaining: 3.32s
116:	learn: 21721.6396334	total: 439ms	remaining: 3.32s
117:	learn: 21625.0328581	total: 443ms	remaining:

276:	learn: 14996.9549460	total: 967ms	remaining: 2.52s
277:	learn: 14950.1109979	total: 971ms	remaining: 2.52s
278:	learn: 14929.3693481	total: 974ms	remaining: 2.52s
279:	learn: 14895.3075496	total: 977ms	remaining: 2.51s
280:	learn: 14870.0986536	total: 980ms	remaining: 2.51s
281:	learn: 14835.8304573	total: 984ms	remaining: 2.5s
282:	learn: 14823.2403903	total: 987ms	remaining: 2.5s
283:	learn: 14805.3664178	total: 990ms	remaining: 2.5s
284:	learn: 14784.4974638	total: 993ms	remaining: 2.49s
285:	learn: 14770.1105376	total: 996ms	remaining: 2.49s
286:	learn: 14750.8358142	total: 999ms	remaining: 2.48s
287:	learn: 14705.1896591	total: 1s	remaining: 2.48s
288:	learn: 14667.5178494	total: 1s	remaining: 2.47s
289:	learn: 14660.5489212	total: 1.01s	remaining: 2.47s
290:	learn: 14631.0790933	total: 1.01s	remaining: 2.46s
291:	learn: 14603.6771511	total: 1.01s	remaining: 2.46s
292:	learn: 14580.8455709	total: 1.02s	remaining: 2.46s
293:	learn: 14553.5846926	total: 1.02s	remaining: 2.46s
2

436:	learn: 11156.3686819	total: 1.56s	remaining: 2.01s
437:	learn: 11145.6710388	total: 1.56s	remaining: 2s
438:	learn: 11128.4241022	total: 1.57s	remaining: 2s
439:	learn: 11101.3877386	total: 1.57s	remaining: 2s
440:	learn: 11080.3859934	total: 1.58s	remaining: 2s
441:	learn: 11057.2864352	total: 1.58s	remaining: 2s
442:	learn: 11040.6879467	total: 1.58s	remaining: 1.99s
443:	learn: 11006.4243934	total: 1.59s	remaining: 1.99s
444:	learn: 10992.4502820	total: 1.59s	remaining: 1.98s
445:	learn: 10970.6814042	total: 1.59s	remaining: 1.98s
446:	learn: 10954.9626851	total: 1.6s	remaining: 1.97s
447:	learn: 10935.9699009	total: 1.6s	remaining: 1.97s
448:	learn: 10922.3750836	total: 1.6s	remaining: 1.97s
449:	learn: 10896.4606975	total: 1.6s	remaining: 1.96s
450:	learn: 10880.9257382	total: 1.61s	remaining: 1.96s
451:	learn: 10871.0786985	total: 1.61s	remaining: 1.95s
452:	learn: 10851.4553055	total: 1.61s	remaining: 1.95s
453:	learn: 10828.5471675	total: 1.62s	remaining: 1.95s
454:	learn:

615:	learn: 8477.5281517	total: 2.15s	remaining: 1.34s
616:	learn: 8460.1022842	total: 2.15s	remaining: 1.34s
617:	learn: 8439.5506011	total: 2.16s	remaining: 1.33s
618:	learn: 8427.3783064	total: 2.16s	remaining: 1.33s
619:	learn: 8418.9280738	total: 2.16s	remaining: 1.33s
620:	learn: 8407.8842994	total: 2.17s	remaining: 1.32s
621:	learn: 8390.3061576	total: 2.17s	remaining: 1.32s
622:	learn: 8377.9414414	total: 2.17s	remaining: 1.31s
623:	learn: 8366.2413975	total: 2.18s	remaining: 1.31s
624:	learn: 8350.0985074	total: 2.18s	remaining: 1.31s
625:	learn: 8340.1883659	total: 2.18s	remaining: 1.3s
626:	learn: 8324.2359898	total: 2.19s	remaining: 1.3s
627:	learn: 8308.7884170	total: 2.19s	remaining: 1.3s
628:	learn: 8296.4018383	total: 2.19s	remaining: 1.29s
629:	learn: 8283.0881353	total: 2.2s	remaining: 1.29s
630:	learn: 8274.3047841	total: 2.2s	remaining: 1.29s
631:	learn: 8265.5351978	total: 2.2s	remaining: 1.28s
632:	learn: 8250.6240584	total: 2.21s	remaining: 1.28s
633:	learn: 8238

...

176:	learn: 18430.0990949	total: 576ms	remaining: 2.67s
177:	learn: 18379.0919174	total: 580ms	remaining: 2.68s
178:	learn: 18295.5987972	total: 583ms	remaining: 2.67s
179:	learn: 18262.8868765	total: 586ms	remaining: 2.67s
180:	learn: 18217.3177770	total: 589ms	remaining: 2.67s
181:	learn: 18154.9667166	total: 593ms	remaining: 2.66s
182:	learn: 18117.5258079	total: 596ms	remaining: 2.66s
183:	learn: 18067.1790769	total: 599ms	remaining: 2.65s
184:	learn: 18041.7584735	total: 602ms	remaining: 2.65s
185:	learn: 18013.0673001	total: 605ms	remaining: 2.65s
186:	learn: 17965.3339786	total: 608ms	remaining: 2.64s
187:	learn: 17942.5972172	total: 611ms	remaining: 2.64s
188:	learn: 17895.0750483	total: 614ms	remaining: 2.64s
189:	learn: 17858.5180969	total: 618ms	remaining: 2.63s
190:	learn: 17827.8260227	total: 621ms	remaining: 2.63s
191:	learn: 17780.4586901	total: 624ms	remaining: 2.63s
192:	learn: 17725.2958793	total: 627ms	remaining: 2.62s
193:	learn: 17695.8061137	total: 630ms	remaining

359:	learn: 12767.0449466	total: 1.17s	remaining: 2.07s
360:	learn: 12737.6163620	total: 1.17s	remaining: 2.07s
361:	learn: 12714.2742058	total: 1.17s	remaining: 2.06s
362:	learn: 12690.3976198	total: 1.18s	remaining: 2.06s
363:	learn: 12657.8320694	total: 1.18s	remaining: 2.06s
364:	learn: 12637.5205096	total: 1.18s	remaining: 2.06s
365:	learn: 12613.9597564	total: 1.18s	remaining: 2.05s
366:	learn: 12596.8895615	total: 1.19s	remaining: 2.05s
367:	learn: 12557.2391793	total: 1.19s	remaining: 2.04s
368:	learn: 12547.2111595	total: 1.19s	remaining: 2.04s
369:	learn: 12519.9726910	total: 1.2s	remaining: 2.04s
370:	learn: 12501.4049520	total: 1.2s	remaining: 2.04s
371:	learn: 12468.3899515	total: 1.2s	remaining: 2.03s
372:	learn: 12454.4267984	total: 1.21s	remaining: 2.03s
373:	learn: 12439.3647025	total: 1.21s	remaining: 2.03s
374:	learn: 12425.4128280	total: 1.21s	remaining: 2.02s
375:	learn: 12409.6248718	total: 1.22s	remaining: 2.02s
376:	learn: 12393.0053430	total: 1.22s	remaining: 2

538:	learn: 9530.4419007	total: 1.75s	remaining: 1.5s
539:	learn: 9516.6066359	total: 1.75s	remaining: 1.49s
540:	learn: 9503.2342735	total: 1.76s	remaining: 1.49s
541:	learn: 9492.7189957	total: 1.76s	remaining: 1.49s
542:	learn: 9483.0738211	total: 1.76s	remaining: 1.48s
543:	learn: 9465.2308362	total: 1.77s	remaining: 1.48s
544:	learn: 9439.7840802	total: 1.77s	remaining: 1.48s
545:	learn: 9424.3590549	total: 1.77s	remaining: 1.47s
546:	learn: 9397.4068424	total: 1.78s	remaining: 1.47s
547:	learn: 9392.4406064	total: 1.78s	remaining: 1.47s
548:	learn: 9372.0017699	total: 1.78s	remaining: 1.46s
549:	learn: 9353.7955366	total: 1.78s	remaining: 1.46s
550:	learn: 9335.2088743	total: 1.79s	remaining: 1.46s
551:	learn: 9323.1246071	total: 1.79s	remaining: 1.45s
552:	learn: 9308.0720105	total: 1.79s	remaining: 1.45s
553:	learn: 9293.2438589	total: 1.8s	remaining: 1.45s
554:	learn: 9269.8631874	total: 1.8s	remaining: 1.44s
555:	learn: 9260.7423982	total: 1.8s	remaining: 1.44s
556:	learn: 92

718:	learn: 7344.2498992	total: 2.33s	remaining: 909ms
719:	learn: 7332.1687461	total: 2.33s	remaining: 906ms
720:	learn: 7317.3262322	total: 2.33s	remaining: 903ms
721:	learn: 7312.8186839	total: 2.34s	remaining: 900ms
722:	learn: 7301.7203857	total: 2.34s	remaining: 897ms
723:	learn: 7291.5035790	total: 2.34s	remaining: 893ms
724:	learn: 7282.0227188	total: 2.35s	remaining: 890ms
725:	learn: 7272.2354751	total: 2.35s	remaining: 887ms
726:	learn: 7263.2581632	total: 2.35s	remaining: 884ms
727:	learn: 7258.1935074	total: 2.36s	remaining: 880ms
728:	learn: 7250.5779167	total: 2.36s	remaining: 877ms
729:	learn: 7249.8324377	total: 2.36s	remaining: 874ms
730:	learn: 7236.9057432	total: 2.37s	remaining: 870ms
731:	learn: 7228.2098265	total: 2.37s	remaining: 867ms
732:	learn: 7219.1954340	total: 2.37s	remaining: 864ms
733:	learn: 7207.9800358	total: 2.37s	remaining: 861ms
734:	learn: 7203.1886182	total: 2.38s	remaining: 857ms
735:	learn: 7195.8737454	total: 2.38s	remaining: 854ms
736:	learn

898:	learn: 5864.8557455	total: 2.92s	remaining: 328ms
899:	learn: 5859.0674281	total: 2.92s	remaining: 324ms
900:	learn: 5847.2315332	total: 2.92s	remaining: 321ms
901:	learn: 5841.5682282	total: 2.93s	remaining: 318ms
902:	learn: 5828.5640313	total: 2.93s	remaining: 315ms
903:	learn: 5821.8906620	total: 2.93s	remaining: 311ms
904:	learn: 5814.8605105	total: 2.94s	remaining: 308ms
905:	learn: 5807.0923383	total: 2.94s	remaining: 305ms
906:	learn: 5801.5963654	total: 2.94s	remaining: 302ms
907:	learn: 5793.2576014	total: 2.95s	remaining: 299ms
908:	learn: 5785.6219637	total: 2.95s	remaining: 295ms
909:	learn: 5779.1201547	total: 2.95s	remaining: 292ms
910:	learn: 5773.2090474	total: 2.96s	remaining: 289ms
911:	learn: 5763.1592354	total: 2.96s	remaining: 285ms
912:	learn: 5760.7749414	total: 2.96s	remaining: 282ms
913:	learn: 5751.8380639	total: 2.96s	remaining: 279ms
914:	learn: 5744.6146155	total: 2.97s	remaining: 276ms
915:	learn: 5738.6311699	total: 2.97s	remaining: 272ms
916:	learn